# DataCenter Sentinel — Iteración 1: Simulación del robot

**Qué hace este notebook:**
1. Instala ROS 2 Humble + Gazebo Classic en Colab
2. Clona el repositorio y construye el paquete `sentinel_sim`
3. Genera el mapa del datacenter a partir del layout JSON
4. Lanza la simulación headless (Gazebo + Nav2 + nodos Sentinel)
5. Graba el patrullaje en video con ffmpeg
6. Copia los outputs a Google Drive

**Runtime:** CPU standard de Colab (~12 GB RAM). Sin GPU requerida.

**Tiempo estimado:** 20–30 min (la mayor parte es la instalación de ROS 2).

---
> ⚠️ **Importante:** Ejecutá las celdas **en orden**. Si la sesión se cae,
> empezá desde la celda 0 nuevamente.

## 0 · Verificar entorno

In [ ]:
import subprocess, sys, platform

print(f'Python  : {sys.version}')
print(f'OS      : {platform.platform()}')

# Check Ubuntu version — ROS 2 Humble requires 22.04 (Jammy)
result = subprocess.run(['lsb_release', '-a'], capture_output=True, text=True)
print(result.stdout)

# Check available RAM
mem = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(mem.stdout)

## 1 · Instalar ROS 2 Humble + Gazebo Classic

Esta celda tarda entre **10 y 20 minutos** en un runtime fresco.
Si ROS 2 ya está instalado (sesión reutilizada), se saltea automáticamente.

In [ ]:
import os, subprocess, glob

def run(cmd, **kwargs):
    result = subprocess.run(cmd, shell=True, text=True, capture_output=False, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')

# ── ROS 2 Humble ──────────────────────────────────────────────────────
if os.path.exists('/opt/ros/humble'):
    print('ROS 2 Humble already installed — skipping apt install.')
else:
    print('Installing ROS 2 Humble...')
    run('apt-get update -qq')
    run('apt-get install -y -qq software-properties-common curl gnupg2 lsb-release')
    run('curl -sSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key '
        '-o /usr/share/keyrings/ros-archive-keyring.gpg')
    run('echo "deb [arch=$(dpkg --print-architecture) '
        'signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] '
        'http://packages.ros.org/ros2/ubuntu '
        '$(. /etc/os-release && echo $UBUNTU_CODENAME) main" '
        '| tee /etc/apt/sources.list.d/ros2.list > /dev/null')
    run('apt-get update -qq')
    packages = ' '.join([
        'ros-humble-ros-base',
        'ros-humble-navigation2',
        'ros-humble-nav2-bringup',
        'ros-humble-gazebo-ros-pkgs',
        'ros-humble-robot-state-publisher',
        'ros-humble-joint-state-publisher',
        'ros-humble-tf2-ros',
        'ros-humble-tf2-tools',
        'python3-colcon-common-extensions',
        'python3-rosdep',
        'python3-pip',
        'xvfb',
        'ffmpeg',
        'x11-utils',
    ])
    run(f'apt-get install -y -qq {packages}')
    print('✓ ROS 2 Humble installed.')

# ── Fix Python version mismatch (3.12 vs 3.10) ────────────────────────
# ROS 2 Humble was compiled for Python 3.10; Colab uses 3.12.
# Patch the shebang on ALL Python scripts under /opt/ros/humble/
# (bin/, lib/, libexec/ — includes ros2, spawn_entity.py, and all nodes)
run('apt-get install -y -qq python3.10 python3.10-dev')

patched = 0
search_dirs = ['/opt/ros/humble/bin', '/opt/ros/humble/lib', '/opt/ros/humble/libexec']
for search_dir in search_dirs:
    for path in glob.glob(f'{search_dir}/**', recursive=True):
        if not os.path.isfile(path):
            continue
        try:
            with open(path, 'rb') as f:
                head = f.read(50)
            if b'python3' not in head or b'3.10' in head:
                continue
            with open(path, 'r', errors='replace') as f:
                content = f.read()
            new = content.replace('#!/usr/bin/env python3', '#!/usr/bin/python3.10', 1)
            new = new.replace('#!/usr/bin/python3\n', '#!/usr/bin/python3.10\n', 1)
            if new != content:
                with open(path, 'w') as f:
                    f.write(new)
                patched += 1
        except Exception:
            pass

print(f'✓ Patched {patched} ROS 2 scripts → python3.10')
v = subprocess.run('python3.10 --version', shell=True, capture_output=True, text=True)
print(f'  python3.10: {v.stdout.strip()}')

os.environ['DISPLAY'] = ':99'
print('✓ Environment ready.')

## 2 · Clonar repositorio y construir el paquete

In [ ]:
import os, subprocess

REPO_URL  = 'https://github.com/fedeferreyra98/datacenter-sentinel.git'  
REPO_BRANCH = 'feature/simulacion-robot-mvp'
WS_DIR    = '/root/sentinel_ws'
SRC_DIR   = f'{WS_DIR}/src'
REPO_DIR  = f'{SRC_DIR}/datacenter-sentinel'

def run_ws(cmd):
    """Run command sourcing ROS 2 setup first."""
    full = f'source /opt/ros/humble/setup.bash && {cmd}'
    result = subprocess.run(full, shell=True, executable='/bin/bash',
                            text=True, capture_output=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Failed: {cmd}')

os.makedirs(SRC_DIR, exist_ok=True)

if os.path.exists(REPO_DIR):
    print('Repo already cloned — fetching and checking out branch.')
    subprocess.run(f'git -C {REPO_DIR} fetch origin', shell=True, check=True)
    subprocess.run(f'git -C {REPO_DIR} checkout {REPO_BRANCH}', shell=True, check=True)
    subprocess.run(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}', shell=True, check=True)
else:
    print(f'Cloning {REPO_URL} (branch: {REPO_BRANCH})...')
    subprocess.run(
        f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}',
        shell=True, check=True
    )

# Verify the key file exists before proceeding
map_script = f'{REPO_DIR}/sim/scripts/generate_map.py'
assert os.path.exists(map_script), f'File not found after clone: {map_script}'

# Generate the Nav2 map from layout JSON
print('Generating occupancy map...')
run_ws(f'python3 {map_script}')

os.makedirs(f'{REPO_DIR}/sim/maps', exist_ok=True)

# Build the sentinel_sim package
print('Building sentinel_sim...')
run_ws(f'cd {WS_DIR} && colcon build --packages-select sentinel_sim --symlink-install')
print('✓ Build complete.')
# ── Patch shebangs on installed sentinel_sim executables ─────────────
# colcon build generates node scripts with #!/usr/bin/env python3 (→ 3.12).
# We need them to use python3.10 like the rest of ROS 2.
import glob
patched = []
for path in glob.glob(f'{WS_DIR}/install/sentinel_sim/lib/sentinel_sim/*'):
    if not os.path.isfile(path):
        continue
    with open(path, 'r') as f:
        content = f.read()
    if not content.startswith('#!'):
        continue
    first_line = content.split('\n')[0]
    if 'python' in first_line and '3.10' not in first_line:
        content = content.replace('#!/usr/bin/env python3', '#!/usr/bin/python3.10', 1)
        content = content.replace('#!/usr/bin/python3\n', '#!/usr/bin/python3.10\n', 1)
        with open(path, 'w') as f:
            f.write(content)
        patched.append(os.path.basename(path))
print(f'✓ Patched {len(patched)} sentinel_sim executables: {patched}')


## 3 · Iniciar display virtual (Xvfb) y comenzar grabación

In [ ]:
import subprocess, os, time

OUTPUT_DIR  = '/root/sentinel_outputs'
VIDEO_PATH  = f'{OUTPUT_DIR}/patrol_video.mp4'
CSV_PATH    = f'{OUTPUT_DIR}/patrol_log.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Start Xvfb ──────────────────────────────────────────────────────────
xvfb = subprocess.Popen(
    ['Xvfb', ':99', '-screen', '0', '1280x720x24'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
os.environ['DISPLAY'] = ':99'
time.sleep(2)
print(f'Xvfb PID: {xvfb.pid}')

# ── Start ffmpeg recording ───────────────────────────────────────────────
ffmpeg = subprocess.Popen(
    [
        'ffmpeg', '-y',
        '-f', 'x11grab',
        '-video_size', '1280x720',
        '-framerate', '15',
        '-i', ':99.0',
        '-c:v', 'libx264',
        '-preset', 'ultrafast',
        '-pix_fmt', 'yuv420p',
        VIDEO_PATH,
    ],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print(f'ffmpeg PID: {ffmpeg.pid}')
print(f'Recording to: {VIDEO_PATH}')

# Store PIDs for cleanup cell
PIDS = {'xvfb': xvfb.pid, 'ffmpeg': ffmpeg.pid}

## 4 · Lanzar simulación completa

Esto inicia en background:
- `gzserver` con el mundo del datacenter
- `robot_state_publisher`
- Nav2 stack
- `virtual_sensor_node`, `sensor_logger_node`, `patrol_node`

Esperá a ver el mensaje **"Patrol complete"** en el output antes de continuar.

In [ ]:
import subprocess, os, time

WS_DIR   = '/root/sentinel_ws'
CSV_PATH = '/root/sentinel_outputs/patrol_log.csv'

# ── Limpieza agresiva de procesos y sockets anteriores ────────────────
print('Cleaning up leftover processes...')
subprocess.run('pkill -9 -f gzserver  || true', shell=True)
subprocess.run('pkill -9 -f gzclient  || true', shell=True)
subprocess.run('pkill -9 -f gz        || true', shell=True)
subprocess.run('fuser -k 11345/tcp    || true', shell=True)
# Borrar socket/lock files que Gazebo deja en /tmp y que bloquean el puerto
subprocess.run('rm -f /tmp/.gazebo* /tmp/gazebo* || true', shell=True)
subprocess.run(
    'source /opt/ros/humble/setup.bash && ros2 daemon stop || true',
    shell=True, executable='/bin/bash', capture_output=True
)
time.sleep(5)  # dar tiempo al SO para liberar el puerto
print('Cleanup done.')

# ── Launch ────────────────────────────────────────────────────────────
# headless:=false → lanza gzclient que dibuja en Xvfb → ffmpeg puede grabarlo
# DISPLAY=:99 asegura que gzclient use el display virtual de Xvfb
LAUNCH_CMD = (
    f'export DISPLAY=:99 && '
    f'source /opt/ros/humble/setup.bash && '
    f'source {WS_DIR}/install/setup.bash && '
    f'ros2 launch sentinel_sim simulation.launch.py '
    f'headless:=false '
    f'csv_output:={CSV_PATH} '
    f'2>&1'
)

print('Starting simulation...')
launch_proc = subprocess.Popen(
    LAUNCH_CMD, shell=True, executable='/bin/bash',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True,
)

TIMEOUT_SECS = 600
start = time.time()
patrol_done = False

for line in launch_proc.stdout:
    print(line, end='', flush=True)
    if 'Patrol complete' in line:
        patrol_done = True
        break
    if time.time() - start > TIMEOUT_SECS:
        print('⚠ Timeout reached — stopping simulation.')
        break

print(f'\n✓ Patrol done: {patrol_done}')
print(f'  Elapsed: {time.time() - start:.0f}s')

## 5 · Detener simulación y cerrar grabación

In [ ]:
import subprocess, time

# Terminar el proceso de launch
try:
    launch_proc.terminate()
    launch_proc.wait(timeout=10)
except Exception:
    pass

# Matar todos los procesos relacionados
subprocess.run('pkill -9 -f gzserver  || true', shell=True)
subprocess.run('pkill -9 -f gzclient  || true', shell=True)
subprocess.run('pkill -9 -f gz        || true', shell=True)
subprocess.run('fuser -k 11345/tcp    || true', shell=True)
subprocess.run('pkill -9 -f ros2      || true', shell=True)
subprocess.run('pkill -9 -f patrol_node       || true', shell=True)
subprocess.run('pkill -9 -f virtual_sensor    || true', shell=True)
subprocess.run('pkill -9 -f sensor_logger     || true', shell=True)

# Detener ffmpeg para cerrar el video
try:
    subprocess.run(f'kill -INT {PIDS["ffmpeg"]}', shell=True)
    time.sleep(3)
    subprocess.run(f'kill -9 {PIDS["xvfb"]}', shell=True)
except Exception:
    pass

print('Simulation stopped. Outputs:')
subprocess.run('ls -lh /root/sentinel_outputs/', shell=True)

## 6 · Inspeccionar resultados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # headless backend

CSV_PATH = '/root/sentinel_outputs/patrol_log.csv'

df = pd.read_csv(CSV_PATH)
print(f'Rows in patrol_log.csv: {len(df)}')
print(df.to_string(index=False))

# ── Plot sensor readings per rack ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('DataCenter Sentinel — Patrol Sensor Readings', fontsize=14)

metrics = [
    ('temperature_c', 'Temperature (°C)', 'tomato'),
    ('humidity_pct',  'Humidity (%)',      'steelblue'),
    ('power_draw_kw', 'Power Draw (kW)',   'mediumseagreen'),
    ('noise_db',      'Noise Level (dB)',  'mediumpurple'),
]

for ax, (col, label, color) in zip(axes.flat, metrics):
    bars = ax.bar(df['rack_id'], df[col], color=color, alpha=0.8)
    ax.set_title(label)
    ax.set_xlabel('Rack')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)

    # Highlight anomalous readings
    for i, (_, row) in enumerate(df.iterrows()):
        if row.get('anomaly_injected', False):
            bars[i].set_edgecolor('red')
            bars[i].set_linewidth(2.5)

plt.tight_layout()
chart_path = '/root/sentinel_outputs/sensor_chart.png'
plt.savefig(chart_path, dpi=120)
print(f'Chart saved to {chart_path}')
plt.show()

## 7 · Copiar outputs a Google Drive

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/datacenter-sentinel/iter01'
os.makedirs(DRIVE_DIR, exist_ok=True)

files_to_copy = [
    '/root/sentinel_outputs/patrol_log.csv',
    '/root/sentinel_outputs/patrol_video.mp4',
    '/root/sentinel_outputs/sensor_chart.png',
]

for src in files_to_copy:
    if os.path.exists(src):
        dest = os.path.join(DRIVE_DIR, os.path.basename(src))
        shutil.copy2(src, dest)
        size = os.path.getsize(dest) / 1024
        print(f'✓ {os.path.basename(src)} → Drive ({size:.1f} KB)')
    else:
        print(f'⚠ Not found: {src}')

print(f'\nOutputs saved to: {DRIVE_DIR}')

---
## Troubleshooting

**Gazebo se cuelga / RAM insuficiente**
```bash
# Verificar uso de memoria durante la simulación
!free -h
# Si la RAM supera el 90%, reducir el número de racks en datacenter_layout.json
```

**Nav2 no arranca**
- Verificar que el transform `map → odom` esté publicado: `ros2 run tf2_tools view_frames`
- Aumentar el delay del patrol_node en simulation.launch.py (default: 20s)

**El robot no navega / colisiona**
- Revisar que los waypoints en `waypoints.yaml` estén en el pasillo central (y=4.5–5.5)
- Reducir `max_vel_x` en `nav2_params.yaml` a 0.15 para espacios más ajustados

**ffmpeg / video no se genera**
- Verificar que Xvfb esté corriendo: `ps aux | grep Xvfb`
- Re-ejecutar la celda 3 antes de lanzar la simulación

**Colab no tiene Ubuntu 22.04 (Humble requiere Jammy)**
- En runtimes con Ubuntu 20.04, usar ROS 2 Galactic en lugar de Humble
  (reemplazar `humble` por `galactic` en todos los comandos apt)